In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_5.pickle

In [ ]:
%%cudf.pandas.profile
### cell 5 ###

# 1. Filter, compute year & volume, project only needed columns
li = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= "1995-01-01") &
        (lineitem.L_SHIPDATE <  "1997-01-01")
    ]
    .assign(
        L_YEAR=lambda df: df.L_SHIPDATE.dt.year,
        VOLUME=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT)
    )[['L_ORDERKEY','L_SUPPKEY','L_YEAR','VOLUME']]
)

# 2. Pre-filter nation table
nations = nation[
    nation.N_NAME.isin(["FRANCE","GERMANY"])
][['N_NATIONKEY','N_NAME']]

# 3. Attach nation to customer & supplier with minimal columns
cust = (
    customer[['C_CUSTKEY','C_NATIONKEY']]
    .merge(nations, left_on='C_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['C_CUSTKEY','N_NAME']]
    .rename(columns={'N_NAME':'CUST_NATION'})
)

supp = (
    supplier[['S_SUPPKEY','S_NATIONKEY']]
    .merge(nations, left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['S_SUPPKEY','N_NAME']]
    .rename(columns={'N_NAME':'SUPP_NATION'})
)

# 4. Join lineitem → orders → customer → supplier, then filter
df = (
    li
    .merge(orders[['O_ORDERKEY','O_CUSTKEY']], left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')
    .merge(cust, left_on='O_CUSTKEY', right_on='C_CUSTKEY', how='inner')
    .merge(supp, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    [['SUPP_NATION','CUST_NATION','L_YEAR','VOLUME']]
)

# 5. Keep only France↔Germany pairs
df = df[
    ((df.SUPP_NATION == 'FRANCE') & (df.CUST_NATION == 'GERMANY')) |
    ((df.SUPP_NATION == 'GERMANY') & (df.CUST_NATION == 'FRANCE'))
]

# 6. Aggregate on GPU and rename
total = (
    df
    .groupby(['SUPP_NATION','CUST_NATION','L_YEAR'], as_index=False)['VOLUME']
    .sum()
    .rename(columns={'VOLUME':'REVENUE'})
)